In [ ]:
# dataset: structured dataset Amazon–Google_1 (https://github.com/anhaidgroup/sparkly)

# Some simple experiments with the Amazon–Google_1 dataset

import pandas as pd

dataset_path ="../amazon_google"

df_amazon = pd.read_parquet("../amazon_google/table_a.parquet")
df_google = pd.read_parquet("../amazon_google/table_b.parquet")
df_gold = pd.read_parquet("../amazon_google/gold.parquet")

# use only the attributes to compare tuples
df_amazon_only_3_att = df_amazon[['title', 'manufacturer', 'price']] 
df_google_only_3_att = df_google[['title', 'manufacturer', 'price']]

In [18]:
print(df_amazon.columns)
print(df_amazon.shape)
print(df_google.shape)
print(df_gold.shape)

Index(['_id', 'title', 'description', 'manufacturer', 'price'], dtype='object')
(1363, 5)
(3226, 5)
(1300, 2)


In [16]:
# full column width display for the Amazon dataset

with pd.option_context('display.max_colwidth', None):
    display(df_amazon_only_3_att.head())
    display(df_google_only_3_att.head())
    display(df_gold.head())
    display(df_amazon[df_amazon["_id"]==0])
    display(df_google[df_google["_id"]==1878])

,title,manufacturer,price
0,clickart 950 000 premier image pack ( dvd-rom ),broderbund,NaN
1,ca international arcserve lap/desktop oem 30pk,computer associates,NaN
2,noah 's ark activity center ( jewel case ages 3-8 ),victory multimedia,NaN
3,peachtree by sage premium accounting for nonprofits 2007,sage software,599.99
4,singing coach unlimited,carry-a-tune technologies,99.99


,title,manufacturer,price
0,learning quickbooks 2007,intuit,38.99
1,superstart ! fun with reading & writing !,<NA>,8.49
2,qb pos 6.0 basic software,intuit,637.99
3,math missions : the amazing arcade adventure ( grades 3-5 ),<NA>,12.95
4,production prem cs3 mac upgrad,adobe software,805.99


,id1,id2
0,0,1878
1,2,1881
2,3,1879
3,4,1905
4,6,1923


,_id,title,description,manufacturer,price
0,0,clickart 950 000 premier image pack ( dvd-rom ),<NA>,broderbund,NaN


,_id,title,description,manufacturer,price
1878,1878,clickart 950000 premier image pack ( dvd-rom ),massive collection of images & fonts for all your design needs ondvd-rom ! product informationinspire your creativity and perfect any creative project with thousands ofworld-class images in virtually every style . plus clickart 950000 makes iteasy for,<NA>,48.95


In [19]:
# select only id2 that appear more than once in the gold dataset
id2_counts = df_gold['id2'].value_counts()
id2_more_than_once = id2_counts[id2_counts > 1].index
df_gold_filtered = df_gold[df_gold['id2'].isin(id2_more_than_once)]
print(df_gold_filtered.shape)

(18, 2)


In [20]:
df_gold_filtered

,id1,id2
99,107,1696
172,271,1512
211,352,2871
212,353,2871
225,366,2770
227,369,2770
295,471,145
296,471,750
306,484,2807
347,527,2807


Some simple retrieval experiment

In [ ]:
import math

def tf_bm25(term: str, document: str, counted_plural: bool, k1: float, b: float, avg_doc_length: float) -> float:
    # bigger k1 -> smaller tf -> term less important
    # b: scaling factor for doc length compared to avg doc length
    # -> bigger b -> Length Normalization more important 
    doc_length = len(document.split())
    map = {}
    freq_t_d = 0

    for word in document.split():
        if word in map:
            map[word] += 1
        else:
            map[word] = 1

    if counted_plural:
        freq_t_d = (map.get(term, 0) + map.get(term + "s", 0))
    else:
        freq_t_d = map.get(term, 0)

    tf = freq_t_d*(k1+1) / (freq_t_d + k1 * (1 - b + b * doc_length / avg_doc_length))
    
    return tf

def idf_bm25(term: str, documents: list, counted_plural: bool) -> float:
    N = len(documents)
    count = 0

    for document in documents:
        if counted_plural:
            if term in document.split() or term + "s" in document.split():
                count += 1
        else:
            if term in document.split():
                count += 1

    idf = 0
    if count > 0:
        idf = math.log((N - count + 0.5) / (count + 0.5) + 1) # https://pages.cs.wisc.edu/~anhai/papers1/sparkly-vldb2023.pdf

    return idf

def bm25(query: str, document: str, documents: list, counted_plural: bool = False, k1: float = 1.5, b: float = 0.75) -> float:
    avg_doc_length = sum(len(doc.split()) for doc in documents) / len(documents)

    score = 0
    for term in query.split():
        score += tf_bm25(term, document, counted_plural, k1, b, avg_doc_length) * idf_bm25(term, documents, counted_plural)

    return score

In [32]:
def top_k(
    query: str,
    documents: list[str],
    k: int = 10,
    counted_plural: bool = False,
    k1: float = 1.5,
    b: float = 0.75,
) -> list[dict]:
    if k <= 0 or not documents:
        return []

    # Avoid division by zero.
    if not any(document.split() for document in documents):
        return []

    results = pd.DataFrame(columns=["document_index", "document", "score"])
    
    for index, document in enumerate(documents):
        score = bm25(
            query=query,
            document=document,
            documents=documents,
            counted_plural=counted_plural,
            k1=k1,
            b=b,
        )

        # Use this filter after applying the positive IDF formula.
        if score > 0:
            results = pd.concat([results, pd.DataFrame([{
                "document_index": index,
                "document": document,
                "score": score,
            }])], ignore_index=True)

    # Descending score; original document order breaks ties.
    results = results.sort_values(
        by=["score", "document_index"],
        ascending=[False, True]
    )

    return results.head(k)

In [ ]:
table_A = df_amazon
table_B = df_google

# first query example
documents = table_A["title"].fillna("").str.lower().tolist()
record_ids = table_A["_id"].tolist()

query = str(table_B.iloc[1696]["title"]).lower()

results = top_k(query, documents, k=10)

adobe systems inc 29180248 adobe photoshop elements 5.0 / premiere elements 3.0
True Matches are 546 and 107


/tmp/ipykernel_9596/797859573.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([{


In [40]:
# show full column width of the results DataFrame
print(query)
print("True Matches are 546 and 107")

with pd.option_context('display.max_colwidth', None):
    display(results)

adobe systems inc 29180248 adobe photoshop elements 5.0 / premiere elements 3.0
True Matches are 546 and 107


,document_index,document,score
55,546,adobe photoshop elements 5.0 premiere elements 3.0 bundle,32.936946
28,267,adobe photoshop elements 5.0 / premier elements 3.0 [ lb ],28.699466
45,484,adobe premiere elements 3.0,27.777230
52,527,adobe premiere elements 3.0,27.777230
36,364,adobe premiere elements 3.0 [ lb ],22.066290
9,107,photoshop elements 5 and premiere elements 3 bundle win minibox ( 29180248 ),20.434273
86,853,onone essentials for adobe photoshop elements,19.201628
53,536,adobe photoshop elements 4.0 ( mac ),17.885690
17,157,adobe photoshop elements 4.0 plus adobe premiere elements 2.0 complete package 1 user cd mini-box win canadian french 29180162 ),16.764594
122,1330,onone essentials for adobe photoshop elements version 2 full,15.729695


In [41]:
query_2 = str(table_B.iloc[2770]["title"]).lower()

results_2 = top_k(query_2, documents, k=10)
# show full column width of the results DataFrame

print(query_2)
print("True Matches are 366 and 369")

with pd.option_context('display.max_colwidth', None):
    display(results_2)

/tmp/ipykernel_9596/797859573.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([{


tlc arthur 's kindergarten learning system ( pc ) encore
True Matches are 366 and 369


,document_index,document,score
105,369,tlc arthur 's kindergarten learning system 2008,29.051134
104,366,riverdeep arthur 's kindergarten,18.077471
33,124,tlc dr. seuss reading learning system 2008,14.690353
285,1051,tlc clue finders math learning system sb by riverdeep,12.919534
122,420,student home learning system for pc/mac,9.971618
143,510,reader rabbit learning system 1st grade,9.971618
12,55,jumpstart learning games phonics ( pc & mac ),9.346630
259,932,registry mechanic 5.x pc system repair,8.604100
281,1033,pc professor fuji 's kakuro,8.149037
131,456,clickart fonts ( 2006 ) by encore,7.218341


In [42]:
query_3 = str(table_B.iloc[1512]["title"]).lower()

results_3 = top_k(query_3, documents, k=10)
# show full column width of the results DataFrame

print(query_3)
print("True Matches are 271 and 604")

with pd.option_context('display.max_colwidth', None):
    display(results_3)

/tmp/ipykernel_9596/797859573.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([{


microsoft w87-02380 office sbe 2007 win32
True Matches are 271 and 604


,document_index,document,score
85,578,microsoft office excel 2007,11.815026
34,271,microsoft office small business 2007,10.876699
54,476,ae outlook 2007 win32,10.382074
92,604,microsoft office small business 2007 full version,9.385881
40,348,microsoft office & windows training,7.855614
105,753,microsoft office & windows training professional,7.277638
75,541,microsoft onenote 2007,7.270686
78,547,microsoft outlook 2007,7.270686
94,606,microsoft office 2004 professional ( mac ),6.778882
130,1145,microsoft office 2004 standard ( mac ),6.778882


In [ ]:
# Findings
# To find all true matches:

# Make use of Score-Distribution to select thresholds:

# Example policy 1:
# relative threshold for considering a match: 0.6 (compare to the highest score)
# combine with min threshold for absolute bm_25 score as well e.g: 5% quantile 
# (of all true matches scores, so that 95% of the matches are above this threshold)

# increase k with scaling factor 2^i (i start = 1) and stop 
# increasing when the relative similarity score falls below the threshold
# but the scaling factor should also consider the table size of A, bigger A -> larger scaling factor and vice versa